# 06 - Evolução do baseline: Item2Vec v1 → v2 (Fase 2 · ML)
**Squad 2 — Negócio: ML em Streaming | P2 · Recomendação de produtos no carrinho (Item2Vec)**
**Responsável:** Zaiden Emiliano Segundo Seleme · **Branch:** `feat/squad2-fase2-ml-zaiden`

---

### Objetivo
Evoluir o Item2Vec do notebook 03 (v1) e **provar** que a nova versão é melhor: mais acerto no mesmo hold-out e, por ser um modelo sem rótulo, exemplos de vizinhos e recomendações que fazem mais sentido.

### O que o notebook 05 ensinou e muda a forma de medir
35% das cestas de treino são cópias exatas de outra cesta. Os acertos de todos os modelos vêm das cestas de teste que **repetem** uma cesta do treino; nas cestas **novas**, ninguém supera a popularidade. Por isso cada versão é medida em três frentes:

| Frente | Métrica | O que responde |
|---|---|---|
| Acerto | Hit@5 no hold-out (336 cestas), separado entre cestas repetidas e novas | Acerta mais? O ganho é memória ou generalização? |
| Missão de compra | Acerto de subcategoria nas **cestas novas** | Em cesta nunca vista, sugere algo da mesma missão do item que faltava? |
| Sentido dos vetores | Coerência: % dos 5 vizinhos mais próximos que dividem a categoria raiz (nos 200 SKUs mais vendidos) | Os vizinhos fazem sentido? |

### Regras
* **Toda escolha é feita numa validação dentro do treino** (as últimas 20% cestas do treino, mesmo protocolo do notebook 05). O hold-out só é usado no fim, uma vez, para v1 e v2.
* **Média de 3 seeds por configuração** na validação, e de 10 seeds no hold-out: o Item2Vec varia só pela inicialização.
* **Regra de escolha, fixada antes de rodar:** maior Hit@5 na validação. Se outras configurações ficam a até 1 cesta de diferença (empate dentro do ruído), vence a de maior coerência.

### Entradas e saídas
| Lê (Gold, notebook 02) | Grava (Gold) |
|---|---|
| `item2vec_cestas_treino`, `item2vec_holdout`, `item2vec_skus` | `item2vec_vetores` (o modelo v2, overwrite), `item2vec_recomendacoes_holdout` e `item2vec_avaliacao` (v2 em append, ao lado da v1) |

In [ ]:
%pip install gensim==4.4.0 python-dotenv==1.0.1 --quiet

In [ ]:
dbutils.library.restartPython()

## 0. Ambiente, caminhos e parâmetros

In [ ]:
import os
import math
import hashlib
import itertools
from collections import defaultdict
from datetime import datetime
from dotenv import load_dotenv, find_dotenv
import numpy as np
import pandas as pd
import gensim
from gensim.models import Word2Vec

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed/.env",
        "/Workspace/Repos/estagio_ed/.env",
    ]
    dotenv_path = next((c for c in candidatos if os.path.isfile(c)), None)
load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
assert storage_account and client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

fqdn = f"{storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

GOLD = f"abfss://squad2@{fqdn}/fase2_ml/gold"
TABELAS = {nome: f"{GOLD}/item2vec_{nome}" for nome in
           ["skus", "cestas_treino", "holdout", "vetores", "recomendacoes_holdout", "avaliacao"]}

def ler_delta(caminho):
    return spark.read.format("delta").options(**adls_options).load(caminho)

def gravar_delta(df, destino, modo="overwrite"):
    w = df.write.format("delta").options(**adls_options).mode(modo)
    (w.option("overwriteSchema", "true") if modo == "overwrite" else w.option("mergeSchema", "true")).save(destino)

print(f"gensim {gensim.__version__}")

In [ ]:
VERSAO = "v2"
TOP_K = 5
PREFIXO_SUB = "sub:"   # tokens de subcategoria (etapa 3) começam assim e nunca são recomendados

# v1: os parâmetros do notebook 03, ponto de partida de toda a busca
PARAMS_V1 = {"vector_size": 32, "window": 24, "min_count": 1, "sg": 1,
             "negative": 5, "epochs": 50, "seed": 42, "workers": 1}

GRADE_ETAPA_1 = {"epochs": [50, 100, 200, 400], "ns_exponent": [-0.5, 0.0, 0.5, 0.75, 1.0]}
GRADE_ETAPA_2 = {"vector_size": [16, 32, 64], "negative": [5, 15]}
SEEDS_VALIDACAO = [1, 2, 3]
SEEDS_HOLDOUT = list(range(1, 11))
FRACAO_VALIDACAO = 0.2
N_COERENCIA = 200   # SKUs mais vendidos usados na medida de coerência

## 1. Leitura e marcação das cestas repetidas
Uma cesta de teste "repete o treino" quando a cesta completa (carrinho + item removido) está contida numa cesta de treino. Mesma definição do notebook 05.

In [ ]:
skus = ler_delta(TABELAS["skus"]).toPandas()
treino = ler_delta(TABELAS["cestas_treino"]).orderBy("id_pedido").toPandas()
holdout = ler_delta(TABELAS["holdout"]).orderBy("id_pedido").toPandas()

In [ ]:
cestas_treino = [list(s) for s in treino["skus"]]
subcategoria_de = dict(zip(skus["sku"], skus["subcategoria"]))
raiz_de = dict(zip(skus["sku"], skus["categoria_raiz"]))

def marcar_repetidas(base, cestas):
    conjuntos = [frozenset(c) for c in cestas]
    return np.array([any(frozenset([*c, r]) <= s for s in conjuntos)
                     for c, r in zip(base["carrinho"], base["item_removido"])])

holdout["repete_treino"] = marcar_repetidas(holdout, cestas_treino)
print(f"Treino: {len(cestas_treino):,} cestas | hold-out: {len(holdout):,} cestas, "
      f"{holdout['repete_treino'].sum()} repetem o treino e {(~holdout['repete_treino']).sum()} são novas")

## 2. Peças comuns
* `Vetores`: embrulha os vetores do gensim e esconde os tokens de subcategoria, para que nunca sejam recomendados nem apareçam como vizinhos.
* `recomendar`: **o mesmo método do notebook 03** (cosseno com a média do carrinho + fallback pelas subcategorias do carrinho + mais vendidos). A evolução está no treino; o método de recomendar não muda, para a comparação v1 × v2 medir só o modelo.
* `avaliar`: Hit@5 total, acertos em cestas repetidas e novas, acerto de subcategoria nas novas, cobertura e coerência dos vizinhos.

In [ ]:
class Vetores:
    def __init__(self, wv):
        self.wv = wv
        self.n_tokens_sub = sum(k.startswith(PREFIXO_SUB) for k in wv.index_to_key)

    def __contains__(self, sku):
        return sku in self.wv

    def __getitem__(self, sku):
        return self.wv[sku]

    def skus(self):
        return [k for k in self.wv.index_to_key if not k.startswith(PREFIXO_SUB)]

    def _filtrar(self, pares, topn):
        return [(k, c) for k, c in pares if not k.startswith(PREFIXO_SUB)][:topn]

    def similar_by_vector(self, vetor, topn):
        return self._filtrar(self.wv.similar_by_vector(vetor, topn=topn + self.n_tokens_sub), topn)

    def most_similar(self, sku, topn):
        return self._filtrar(self.wv.most_similar(sku, topn=topn + self.n_tokens_sub), topn)


def com_subcategoria(cestas):
    """Etapa 3: cada cesta ganha as suas subcategorias como 'palavras' extras."""
    return [c + sorted({PREFIXO_SUB + subcategoria_de[s] for s in c}) for c in cestas]


def sem_duplicatas(cestas):
    """Etapa 3: mantém só a primeira ocorrência de cada conjunto de SKUs."""
    vistos, saida = set(), []
    for c in cestas:
        if frozenset(c) not in vistos:
            vistos.add(frozenset(c))
            saida.append(c)
    return saida


def treinar(cestas, params, representacao=None, seed=42):
    frases = representacao(cestas) if representacao else cestas
    return Vetores(Word2Vec(sentences=frases, **{**params, "seed": seed}).wv)


def preparar_contexto(cestas, freq=None):
    if freq is None:
        freq = pd.Series([s for c in cestas for s in c]).value_counts()
    populares = sorted(freq[freq > 0].index, key=lambda s: (-freq[s], s))
    por_sub = defaultdict(list)
    for sku in populares:
        por_sub[subcategoria_de.get(sku)].append(sku)
    return {"mais_vendidos": populares, "mais_vendidos_sub": dict(por_sub)}


def completar(recs, carrinho, candidatos):
    for sku in candidatos:
        if len(recs) >= TOP_K:
            break
        if sku not in carrinho and sku not in recs:
            recs.append(sku)
    return recs


def recomendar(vetores, carrinho, ctx):
    no_carrinho = set(carrinho)
    conhecidos = [s for s in carrinho if s in vetores]
    recs = []
    if conhecidos:
        perfil = np.mean([vetores[s] for s in conhecidos], axis=0)
        for sku, _ in vetores.similar_by_vector(perfil, topn=TOP_K + len(conhecidos)):
            if sku not in no_carrinho:
                recs.append(sku)
            if len(recs) == TOP_K:
                break
    for sub in dict.fromkeys(subcategoria_de.get(s) for s in carrinho):
        recs = completar(recs, no_carrinho, ctx["mais_vendidos_sub"].get(sub, []))
    return completar(recs, no_carrinho, ctx["mais_vendidos"])


def coerencia(vetores, ctx, n=N_COERENCIA, k=5):
    """% dos k vizinhos mais próximos que dividem a categoria raiz, nos n SKUs mais vendidos."""
    alvo = [s for s in ctx["mais_vendidos"] if s in vetores][:n]
    pares = [(s, v) for s in alvo for v, _ in vetores.most_similar(s, topn=k)]
    return np.mean([raiz_de.get(s) == raiz_de.get(v) for s, v in pares])


def avaliar(vetores, base, ctx):
    linhas, recomendados = [], set()
    for cesta in base.itertuples(index=False):
        recs = recomendar(vetores, list(cesta.carrinho), ctx)
        recomendados.update(recs)
        linhas.append({"id_pedido": int(cesta.id_pedido), "recomendacoes": recs, "item_removido": cesta.item_removido,
                       "acerto": cesta.item_removido in recs,
                       "acerto_subcategoria": subcategoria_de.get(cesta.item_removido)
                                              in {subcategoria_de.get(r) for r in recs}})
    df = pd.DataFrame(linhas)
    rep = base["repete_treino"].to_numpy()
    metricas = {"cestas": len(df), "acertos": int(df["acerto"].sum()), "hit_at_5": df["acerto"].mean(),
                "precision_at_5": df["acerto"].mean() / TOP_K,
                "acertos_repetidas": int(df["acerto"][rep].sum()), "acertos_novas": int(df["acerto"][~rep].sum()),
                "acerto_subcategoria": df["acerto_subcategoria"].mean(),
                "acerto_subcategoria_novas": df["acerto_subcategoria"][~rep].mean(),
                "cobertura_catalogo": len(recomendados) / len(skus), "skus_recomendados": len(recomendados),
                "coerencia_vizinhos": coerencia(vetores, ctx)}
    return metricas, df


def wilson(acertos, n, z=1.959963984540054):
    p = acertos / n
    d = 1 + z * z / n
    centro = (p + z * z / (2 * n)) / d
    margem = z / d * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, centro - margem), min(1.0, centro + margem)


def mcnemar_exato(acertos_a, acertos_b):
    so_a, so_b = int(np.sum(acertos_a & ~acertos_b)), int(np.sum(~acertos_a & acertos_b))
    n = so_a + so_b
    if n == 0:
        return so_a, so_b, 1.0
    return so_a, so_b, min(1.0, 2 * sum(math.comb(n, i) for i in range(min(so_a, so_b) + 1)) / 2 ** n)

## 3. Validação dentro do treino
As últimas 20% cestas do treino (por instante do pedido) viram validação, com 1 item removido por cesta pelo menor `md5(id_pedido|sku)`, a mesma regra do notebook 05. Os modelos da busca treinam só com os 80% iniciais.

In [ ]:
def item_validacao(id_pedido, cesta):
    return min(cesta, key=lambda s: hashlib.md5(f"{id_pedido}|{s}".encode()).hexdigest())

ordenado = treino.sort_values(["dt_pedido", "id_pedido"]).reset_index(drop=True)
corte = int(len(ordenado) * (1 - FRACAO_VALIDACAO))
sub_treino, sub_val = ordenado.iloc[:corte], ordenado.iloc[corte:]
cestas_sub_treino = [list(s) for s in sub_treino["skus"]]

validacao = pd.DataFrame({"id_pedido": sub_val["id_pedido"].values,
                          "item_removido": [item_validacao(i, c) for i, c in zip(sub_val["id_pedido"], sub_val["skus"])]})
validacao["carrinho"] = [sorted(set(c) - {r}) for c, r in zip(sub_val["skus"], validacao["item_removido"])]
validacao["repete_treino"] = marcar_repetidas(validacao, cestas_sub_treino)
ctx_val = preparar_contexto(cestas_sub_treino)
UMA_CESTA = 1 / len(validacao)   # tolerância de empate da regra de escolha
print(f"Validação: {len(cestas_sub_treino):,} cestas para treinar | {len(validacao):,} para validar "
      f"({validacao['repete_treino'].sum()} repetem o sub-treino)")


def medir_na_validacao(nome, params, representacao=None):
    resultados = [avaliar(treinar(cestas_sub_treino, params, representacao, seed=s), validacao, ctx_val)[0]
                  for s in SEEDS_VALIDACAO]
    media = pd.DataFrame(resultados).mean(numeric_only=True).to_dict()
    return {"config": nome, **media}


def escolher(tabela):
    """Maior Hit@5; empates dentro de 1 cesta decididos pela coerência dos vizinhos."""
    melhor_hit = tabela["hit_at_5"].max()
    empatados = tabela[tabela["hit_at_5"] >= melhor_hit - UMA_CESTA - 1e-9]
    return empatados.sort_values("coerencia_vizinhos", ascending=False).iloc[0]


def mostrar(tabela):
    t = tabela.copy()
    for c in ["hit_at_5", "acerto_subcategoria_novas", "coerencia_vizinhos", "cobertura_catalogo"]:
        t[c] = (100 * t[c]).round(2).astype(str) + "%"
    return t[["config", "hit_at_5", "acertos_repetidas", "acertos_novas",
              "acerto_subcategoria_novas", "coerencia_vizinhos", "cobertura_catalogo"]]

## 4. Busca em três etapas
1. **Treino (`epochs` × `ns_exponent`).** Com só ~1 mil frases, 50 épocas podem deixar os vetores subtreinados. O `ns_exponent` controla quanto as amostras negativas favorecem os SKUs populares. Segundo a literatura de Item2Vec para recomendação, é o hiperparâmetro que mais muda o resultado.
2. **Tamanho do modelo (`vector_size` × `negative`)**, a partir do vencedor da etapa 1.
3. **Representação dos dados**, a partir do vencedor da etapa 2:
   * **sem duplicatas:** tira as cestas repetidas do treino, para o modelo não supervalorizar as cópias;
   * **com subcategoria:** cada cesta ganha as suas subcategorias como palavras extras. É uma informação lateral (ideia do Meta-Prod2Vec) que aproxima os SKUs da mesma missão, mesmo quando eles aparecem em poucas cestas.

`window` fica em 24 em todas as etapas: as cestas não têm ordem, então a janela precisa cobrir a cesta inteira.

In [ ]:
inicio = datetime.now()
etapa1 = pd.DataFrame([medir_na_validacao(f"epochs={ep}, ns_exponent={ns}", {**PARAMS_V1, "epochs": ep, "ns_exponent": ns})
                       for ep, ns in itertools.product(GRADE_ETAPA_1["epochs"], GRADE_ETAPA_1["ns_exponent"])])
display(mostrar(etapa1))
vencedor1 = escolher(etapa1)
ep, ns = vencedor1["config"].replace("epochs=", "").replace(" ns_exponent=", "").split(",")
PARAMS_ETAPA_1 = {**PARAMS_V1, "epochs": int(ep), "ns_exponent": float(ns)}
print(f"Etapa 1 -> {vencedor1['config']} (Hit@5 {vencedor1['hit_at_5']:.2%}) "
      f"[{(datetime.now() - inicio).total_seconds():.0f}s]")

In [ ]:
etapa2 = pd.DataFrame([medir_na_validacao(f"vector_size={vs}, negative={neg}", {**PARAMS_ETAPA_1, "vector_size": vs, "negative": neg})
                       for vs, neg in itertools.product(GRADE_ETAPA_2["vector_size"], GRADE_ETAPA_2["negative"])])
display(mostrar(etapa2))
vencedor2 = escolher(etapa2)
vs, neg = vencedor2["config"].replace("vector_size=", "").replace(" negative=", "").split(",")
PARAMS_V2 = {**PARAMS_ETAPA_1, "vector_size": int(vs), "negative": int(neg)}
print(f"Etapa 2 -> {vencedor2['config']} (Hit@5 {vencedor2['hit_at_5']:.2%})")

In [ ]:
REPRESENTACOES = {"cestas como estão": None, "sem duplicatas": sem_duplicatas,
                  "com subcategoria": com_subcategoria,
                  "sem duplicatas + com subcategoria": lambda c: com_subcategoria(sem_duplicatas(c))}
etapa3 = pd.DataFrame([medir_na_validacao(nome, PARAMS_V2, f) for nome, f in REPRESENTACOES.items()])
v1_val = medir_na_validacao("v1 (notebook 03)", PARAMS_V1)
display(mostrar(pd.concat([pd.DataFrame([v1_val]), etapa3], ignore_index=True)))
vencedor3 = escolher(etapa3)
REPRESENTACAO_V2 = vencedor3["config"]
print(f"Etapa 3 -> {REPRESENTACAO_V2} (Hit@5 {vencedor3['hit_at_5']:.2%}, coerência {vencedor3['coerencia_vizinhos']:.1%})")
print(f"\nItem2Vec v2 = {PARAMS_V2} + representação '{REPRESENTACAO_V2}'")

## 5. v1 × v2 no hold-out
Agora, e só agora, o hold-out. Os dois modelos treinam no treino completo, com seed 42 (a mesma do notebook 03) e com 10 seeds, para separar o ganho real da sorte da inicialização.

In [ ]:
ctx = preparar_contexto(cestas_treino, freq=skus.set_index("sku")["freq_treino"])
VERSOES = {"v1": (PARAMS_V1, None), "v2": (PARAMS_V2, REPRESENTACOES[REPRESENTACAO_V2])}

modelos, resultados = {}, {}
for nome, (params, representacao) in VERSOES.items():
    modelos[nome] = treinar(cestas_treino, params, representacao, seed=42)
    resultados[nome] = avaliar(modelos[nome], holdout, ctx)

tabela = pd.DataFrame([{"versao": n, **m} for n, (m, _) in resultados.items()])
tabela["ic95"] = [f"{100 * a:.1f}% a {100 * b:.1f}%" for a, b in (wilson(k, n) for k, n in zip(tabela["acertos"], tabela["cestas"]))]
exibir = tabela.copy()
for c in ["hit_at_5", "precision_at_5", "acerto_subcategoria", "acerto_subcategoria_novas", "cobertura_catalogo", "coerencia_vizinhos"]:
    exibir[c] = (100 * exibir[c]).round(2).astype(str) + "%"
display(exibir[["versao", "acertos", "hit_at_5", "ic95", "acertos_repetidas", "acertos_novas", "acerto_subcategoria",
                "acerto_subcategoria_novas", "cobertura_catalogo", "coerencia_vizinhos"]])

so_v2, so_v1, p = mcnemar_exato(resultados["v2"][1]["acerto"].to_numpy(), resultados["v1"][1]["acerto"].to_numpy())
print(f"McNemar v2 × v1: só a v2 acertou {so_v2} cestas, só a v1 acertou {so_v1} -> p = {p:.3f} "
      f"({'diferença real' if p < 0.05 else 'empate estatístico'})")

In [ ]:
variacao = []
for nome, (params, representacao) in VERSOES.items():
    for seed in SEEDS_HOLDOUT:
        m, _ = avaliar(treinar(cestas_treino, params, representacao, seed=seed), holdout, ctx)
        variacao.append({"versao": nome, "seed": seed, **m})
variacao = pd.DataFrame(variacao)
resumo = variacao.groupby("versao").agg(
    hit_medio=("hit_at_5", "mean"), hit_min=("hit_at_5", "min"), hit_max=("hit_at_5", "max"),
    repetidas=("acertos_repetidas", "mean"), novas=("acertos_novas", "mean"),
    subcategoria_novas=("acerto_subcategoria_novas", "mean"), coerencia=("coerencia_vizinhos", "mean"),
    cobertura=("cobertura_catalogo", "mean")).reset_index()
for c in ["hit_medio", "hit_min", "hit_max", "subcategoria_novas", "coerencia", "cobertura"]:
    resumo[c] = (100 * resumo[c]).round(2).astype(str) + "%"
display(resumo)
v2_ganha = (variacao[variacao["versao"] == "v2"]["acertos"].to_numpy() > variacao[variacao["versao"] == "v1"]["acertos"].to_numpy()).sum()
print(f"Mesma seed nas duas versões: a v2 acerta mais em {v2_ganha} de {len(SEEDS_HOLDOUT)} seeds")

## 6. Exemplos: os vizinhos fazem mais sentido?
Os mesmos SKUs do notebook 03, mais alguns de missões claras (pet, bebê, café da manhã). Para cada um, os 5 vizinhos mais próximos na v1 e na v2, com a categoria raiz. Um vizinho bom é da mesma missão de compra; um vizinho ruim é coincidência de uma ou duas cestas.

In [ ]:
PREFIXOS_EXEMPLO = ["desinfetantes", "sacosdelixo", "areiasanitaria", "fraldas", "cafepograocapsula"]
exemplos = []
for prefixo in PREFIXOS_EXEMPLO:
    sku = next((s for s in ctx["mais_vendidos"] if s.startswith(prefixo) and s in modelos["v1"] and s in modelos["v2"]), None)
    if sku is None:
        continue
    for versao in ["v1", "v2"]:
        vizinhos = modelos[versao].most_similar(sku, topn=5)
        exemplos.append({"sku": sku, "raiz": raiz_de.get(sku), "versao": versao,
                         "mesma_raiz": f"{sum(raiz_de.get(v) == raiz_de.get(sku) for v, _ in vizinhos)}/5",
                         "vizinhos": " · ".join(f"{v.split('-')[0]} [{raiz_de.get(v)}]" for v, _ in vizinhos)})
display(pd.DataFrame(exemplos))

In [ ]:
def rotulo(sku):
    return f"{sku}  [{subcategoria_de.get(sku)}]"

df1 = resultados["v1"][1].set_index("id_pedido")
df2 = resultados["v2"][1].set_index("id_pedido")
carrinhos = holdout.set_index("id_pedido")["carrinho"]
novas = holdout.set_index("id_pedido")["repete_treino"].eq(False)

def mostrar_cesta(id_pedido, titulo):
    print(f"=== {titulo} | pedido {id_pedido} | {'cesta nova' if novas[id_pedido] else 'repete o treino'}")
    print("Carrinho:", *[rotulo(s) for s in carrinhos[id_pedido]], sep="\n   ")
    print("Item removido:", rotulo(df1.loc[id_pedido, "item_removido"]))
    for nome, df in [("v1", df1), ("v2", df2)]:
        marca = "ACERTOU" if df.loc[id_pedido, "acerto"] else ("mesma subcategoria" if df.loc[id_pedido, "acerto_subcategoria"] else "errou")
        print(f"{nome} ({marca}):", *[rotulo(s) for s in df.loc[id_pedido, "recomendacoes"]], sep="\n   ")
    print()

# uma cesta repetida que só a v2 acertou e uma cesta nova em que só a v2 acertou a subcategoria
so_v2 = df2.index[df2["acerto"] & ~df1["acerto"]]
sub_so_v2 = df2.index[novas & df2["acerto_subcategoria"] & ~df1["acerto_subcategoria"]]
if len(so_v2):
    mostrar_cesta(so_v2[0], "Só a v2 acertou o SKU")
if len(sub_so_v2):
    mostrar_cesta(sub_so_v2[0], "Cesta nova: só a v2 acertou a missão (subcategoria)")

## 7. Gravação na Gold
* `item2vec_vetores`: passa a guardar o **modelo v2** (overwrite), só os vetores de SKU, sem os tokens de subcategoria. É o que a inferência em streaming carrega. A v1 pode ser recriada pelo notebook 03.
* `item2vec_recomendacoes_holdout` e `item2vec_avaliacao`: a v2 entra em **append**, ao lado das linhas da v1, para a evolução ficar auditável na própria Gold.

In [ ]:
treinado_em = datetime.now()
v2 = modelos["v2"]
vetores = pd.DataFrame({"sku": v2.skus()})
vetores["vetor"] = [v2[s].astype(float).tolist() for s in vetores["sku"]]
vetores["versao"] = VERSAO
vetores["params"] = str({**PARAMS_V2, "representacao": REPRESENTACAO_V2})
vetores["treinado_em"] = treinado_em
gravar_delta(spark.createDataFrame(vetores), TABELAS["vetores"])

recs = resultados["v2"][1][["id_pedido", "recomendacoes", "item_removido", "acerto"]].copy()
recs["modelo"] = "item2vec_com_fallback"
recs["versao"] = VERSAO
gravar_delta(spark.createDataFrame(recs), TABELAS["recomendacoes_holdout"], modo="append")

m = resultados["v2"][0]
aval = pd.DataFrame([{"modelo": "item2vec_com_fallback", "cestas": m["cestas"], "hit_at_5": m["hit_at_5"],
                      "precision_at_5": m["precision_at_5"], "cobertura_catalogo": m["cobertura_catalogo"],
                      "skus_recomendados": m["skus_recomendados"], "acerto_subcategoria": m["acerto_subcategoria"],
                      "versao": VERSAO, "params": str({**PARAMS_V2, "representacao": REPRESENTACAO_V2}),
                      "avaliado_em": treinado_em}])
gravar_delta(spark.createDataFrame(aval), TABELAS["avaliacao"], modo="append")

display(ler_delta(TABELAS["avaliacao"]).filter("modelo = 'item2vec_com_fallback'")
        .select("versao", "hit_at_5", "cobertura_catalogo", "acerto_subcategoria", "avaliado_em").orderBy("avaliado_em").toPandas())

## 8. Resultado

> Execução no Databricks em 23/11/2026: a busca escolheu a mesma v2 e os acertos são idênticos aos da execução local (v1 14 × v2 22 com seed 42, as mesmas 10 seeds). Só cobertura e coerência variam na segunda casa decimal (por exemplo, cobertura da v2 com seed 42 de 18,63% local para 18,69%), por diferença de aritmética de ponto flutuante entre as máquinas. A v2 foi gravada em `item2vec_vetores`, e `item2vec_avaliacao` agora tem as linhas `v1_baseline` e `v2`.

### A v2
`epochs` 200 (era 50), `ns_exponent` 1,0 (era 0,75), `negative` 15 (era 5) e **subcategoria como palavra extra em cada cesta**. `vector_size` 32, `window` 24 e o jeito de recomendar não mudaram. Cada escolha saiu da validação dentro do treino:

| Etapa | O que mudou | Hit@5 na validação | Coerência dos vizinhos |
|---|---|---|---|
| v1 (notebook 03) | — | 10,9% | 41,6% |
| 1. Treino | 200 épocas, `ns_exponent` 1,0 | 19,9% | 35,3% |
| 2. Tamanho do modelo | `negative` 15 (empate com 5, desempatado pela coerência) | 20,0% | 35,6% |
| 3. Representação | + subcategoria como token | **20,1%** | **38,1%** |

Mais épocas foi o que mais pesou no acerto, mas sozinho piorou o sentido dos vetores (coerência de 41,6% para 35,3%). A subcategoria como token recuperou boa parte da coerência sem perder acerto. Remover as cestas duplicadas não ajudou e piorou as cestas novas, então ficou de fora.

### v1 × v2 no hold-out (336 cestas)
| | v1 | v2 |
|---|---|---|
| Hit@5 (seed 42) | 4,17% (14) | **6,55% (22)** |
| IC 95% | 2,5% a 6,9% | 4,4% a 9,7% |
| Hit@5 médio em 10 seeds | 4,40% (de 3,9% a 5,1%) | **6,31% (de 5,9% a 6,9%)** |
| Acertos nas 20 cestas que repetem o treino (média de 10 seeds) | 12,3 | **18,9** |
| Acertos nas 316 cestas novas (média de 10 seeds) | 2,5 | 2,3 |
| Acerto de subcategoria nas cestas novas | 32,7% | 33,0% |
| Coerência dos vizinhos | 41,9% | 40,1% |
| Cobertura do catálogo | 17,6% | 18,8% |

**Leitura:**
1. **A melhora é real e consistente.**
   * Com seed 42, a v2 acerta 8 cestas que a v1 erra, e a v1 não acerta nenhuma que a v2 erre (McNemar p = 0,008).
   * Nas 10 seeds, a v2 ganha da v1 em todas, e a pior seed da v2 (5,95%) fica acima da melhor seed da v1 (5,06%).
   * O Hit@5 médio sobe 43% (de 4,40% para 6,31%) e já fica no mesmo patamar do FP-Growth calibrado do notebook 05 (5,95%), com mais cobertura (18,8% × 11,6%).
2. **O ganho está nas cestas que repetem o treino** (de 12,3 para 18,9 acertos em 20). Nas cestas novas, as duas versões empatam no SKU exato (2,5 × 2,3) e na missão (33%). A v2 lembra melhor as combinações já vistas, mas ainda não generaliza para cestas novas, o mesmo limite que o notebook 05 mostrou para o FP-Growth.
3. **Os vetores continuam fazendo sentido.** A coerência fica praticamente igual à da v1 (40,1% × 41,9%), enquanto só aumentar as épocas teria levado a 35%. Nos exemplos da seção 6:
   * areia sanitária segue com 5/5 vizinhos de Pets;
   * café segue com 4/5 de Matinais;
   * desinfetante passa de 1/5 para 2/5 vizinhos de Limpeza;
   * saco de lixo e fralda continuam com vizinhos fracos: ainda há poucas cestas desses produtos.
4. **Exemplo de recomendação** (seção 6, cesta nova de limpeza): para um carrinho com amaciante, multiuso, papel toalha e sabão, a v1 sugere saco de lixo e inseticida. A v2 sugere desinfetante e multiuso, acertando a subcategoria do item que faltava (desinfetante).

### Próximos passos
1. **Atualizar o one page de comparação:** o Item2Vec v2 (6,31% em média) contra o FP-Growth calibrado (5,95%) e a v2 do Lucas.
2. **Generalizar para cestas novas:** a frente que nenhum modelo resolveu. Os caminhos são mais dados de treino (o histórico da Silver batch, como o Geovany pediu na revisão de 09/11) e levar a questão das cestas repetidas para a squad.
3. **Inferência em streaming (notebook 04):** carregar `item2vec_vetores` (agora a v2) e gravar o top-5 de cada pedido novo.